# Adding Self Reflection Capability to Agent

## 1. Import necessary packages

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict, Literal
from openai.types.chat.chat_completion_message import ChatCompletionMessage
import json

## 2. Instantiate OpenAI client

In [2]:
load_dotenv()

client = OpenAI()

## 3. Define Parameters

In [3]:
model = "gpt-5-nano-2025-08-07"

temperature = 1.0

## 4. Create Memory Class

In [4]:
class Memory:
    def __init__(self):
        self._messages: List[Dict[str, str]] = []

    def add_message(self, role: Literal['user', 'system', 'assistant'], content: str):
        self._messages.append({
            "role": role,
            "content": content
        })

    def get_messages(self) -> List[Dict[str, str]]:
        return self._messages

    def last_message(self) -> None:
        if self._messages:
            return self._messages[-1]

## 5. Define Self Critique Prompt

In [5]:
SELF_CRITIQUE_PROMPT = """
Reflect on your previous response...
Identify any mistakes, areas for improvement, or ways to clarify the answer, making it more concise. 
Provide a revised response if necessary in a Json Output structure:
{
    "original_response": "",
    "revisions_needed": "",
    "updated_response": ""
}
"""

## 6. Define Agent Class

In [6]:
class Agent:
    """A self-reflection AI Agent"""

    def __init__(
        self,
        name:str = "Agent", 
        role:str = "Personal Assistant",
        instructions:str = "Assist the user with their queries",
        model:str = "gpt-5-nano-2025-08-07",
        temperature:float = 1.0,
    ):
        self.name = name
        self.role = role
        self.instructions = instructions
        self.model = model
        self.temperature = temperature

        self.client = OpenAI()

        self.memory = Memory()
        self.memory.add_message(
            role="system",
            content=f"You're an AI Agent, your role is {self.role}, " 
                    f"and you need to {self.instructions}",
        )

        self.critique_prompt = SELF_CRITIQUE_PROMPT
    
    def invoke(self, 
               user_message: str, 
               self_reflection: bool = False, 
               max_iter: int = 1, 
               verbose: bool = False) -> str:
    
        # Rules
        # - Don't allow values less than 1
        # - Don't allow values greater than 3
        # - Max iter is controlled by self_reflection flag. 
        # - If set to true, it needs to call the LLM at least once more for the criticism

        self.memory.add_message(
            role="user",
            content=user_message
        )
        
        if verbose:
            self._log_last_message()

        max_iter = max_iter if max_iter >= 1 else 1
        max_iter = max_iter if max_iter <= 3 else 3
        max_iter = max_iter if self_reflection else 0.5
        
        loops = 2 * max_iter

        for i in range(loops):
            ai_message = self._get_completion(
                messages = self.memory.get_messages()
            )

            self.memory.add_message(
                role = "assistant",
                content = ai_message.content,
            )
            
            if verbose:
                self._log_last_message()

            if i < loops - 1:
                self.memory.add_message(
                    role = "user", 
                    content = self.critique_prompt
                )
                
                if verbose:
                    self._log_last_message()

                ai_message = self._get_completion(
                    messages = self.memory.get_messages()
                )

    def _get_completion(self, messages:List[Dict])-> ChatCompletionMessage:
        response = self.client.chat.completions.create(
            model=self.model,
            temperature=self.temperature,
            messages=messages
        )
        
        return response.choices[0].message

    def _log_last_message(self):
        print(f"### {self.memory.last_message()['role']} message ###\n".upper())
        print(f"{self.memory.last_message()['content']} \n")
        print("\n________________________________________________________________\n")


## 7. Create Agent

In [7]:
agent = Agent()

agent.invoke(
    user_message="Pick only one. Which is most often used hook in React.js?",
    self_reflection=True,
    verbose=True,
)

### USER MESSAGE ###

Pick only one. Which is most often used hook in React.js? 


________________________________________________________________

### ASSISTANT MESSAGE ###

useState 


________________________________________________________________

### USER MESSAGE ###


Reflect on your previous response...
Identify any mistakes, areas for improvement, or ways to clarify the answer, making it more concise. 
Provide a revised response if necessary in a Json Output structure:
{
    "original_response": "",
    "revisions_needed": "",
    "updated_response": ""
}
 


________________________________________________________________

### ASSISTANT MESSAGE ###

{
  "original_response": "useState",
  "revisions_needed": "The claim is broadly correct, but the answer should acknowledge subjectivity and context. 'Most often used' can vary; for many apps useState is the most common built-in hook, but useEffect or custom hooks may appear as often in other projects. A slightly hedged, clarifyi

In [8]:
agent.memory.get_messages()

[{'role': 'system',
  'content': "You're an AI Agent, your role is Personal Assistant, and you need to Assist the user with their queries"},
 {'role': 'user',
  'content': 'Pick only one. Which is most often used hook in React.js?'},
 {'role': 'assistant', 'content': 'useState'},
 {'role': 'user',
  'content': '\nReflect on your previous response...\nIdentify any mistakes, areas for improvement, or ways to clarify the answer, making it more concise. \nProvide a revised response if necessary in a Json Output structure:\n{\n    "original_response": "",\n    "revisions_needed": "",\n    "updated_response": ""\n}\n'},
 {'role': 'assistant',
  'content': '{\n  "original_response": "useState",\n  "revisions_needed": "The claim is broadly correct, but the answer should acknowledge subjectivity and context. \'Most often used\' can vary; for many apps useState is the most common built-in hook, but useEffect or custom hooks may appear as often in other projects. A slightly hedged, clarifying ver

In [9]:
json.loads(agent.memory.last_message()["content"])["updated_response"]

"useState — generally the most frequently used built-in React hook for managing state in functional components. Note that 'most often used' is context-dependent; some codebases rely more on useEffect or on custom hooks."